# 04 · Retrieve — 04 LLM Chunk Scoring (RCS)

**The JSON-parsing and threshold-filtering logic below runs with no API key and no network access; the live-scoring cell near the end needs `GROQ_API_KEY` or `OPENAI_API_KEY` and reports gracefully via `nbio.show_environment()` if neither is loaded.**

Implements "Relevance & Certainty Scoring" (RCS, PaperQA2-style): an LLM
reads every candidate chunk next to the query and returns a `0-10`
relevance score, a one-line summary, and an evidence level. Anything
scoring below a threshold (default `5`) is dropped before the ranking
stage.

**In → out:** the merged, deduplicated list from notebook `03` → the same
list, minus anything the LLM scored too low, each survivor now carrying
`rcs_score`, `summary`, and `evidence_level`.

**What RCS filters, and at what cost:** a chunk that scores `4/10` never
reaches the ranked list -- even if it is the only source that actually
answers a narrow or unusual question. RCS trades recall of borderline-but-
correct material for cutting noise before it reaches whatever downstream
step synthesizes an answer from the shortlist. That tradeoff is invisible
unless you go looking for it, which is what this notebook does: it drops
things on purpose, and shows exactly what got dropped and why -- the
threshold filter (the guardrail) is demonstrated rejecting a bad chunk
before the parsing ladder that feeds it is built out in full.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| The RCS threshold filter (guardrail) | Keeps a chunk only if it parses to a summary and scores `>= SCORE_THRESHOLD`. | one below-threshold chunk rejected, shown first |
| `_parse_json_rcs_output` | Parses a clean/fenced/trailing-comma JSON RCS response. | `_parse_json_rcs_output(sample_responses["clean_json"])` |
| `_extract_summary_and_score_legacy` | Falls back to a plain-text `0-10` / `N/10` scan when JSON parsing fails entirely. | `_extract_summary_and_score_legacy(sample_responses["legacy_plain_text"])` |
| `_parse_rcs_response` | The full ladder: JSON first, legacy plain-text as the last resort. | `_parse_rcs_response(text_out)` → `(summary, score, evidence_level)` |
| `_parse_json_rcs_batch` / `_chunk_from_batch_entry` | Parses `RCS_MODE=batch`'s one-call-scores-many-chunks JSON array format. | `_parse_json_rcs_batch(batch_response)` |
| `score_chunk_live` | Calls a real Groq/OpenAI model with the RCS prompt, if a key is loaded. | `score_chunk_live(question, title, text, provider, client)` |


In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()
nbio.show_environment()

## Step 1 — the guardrail: a chunk that scores too low gets dropped

Before building out the full parsing/repair ladder, prove the actual
guardrail this notebook is about: given a relevance score and a threshold,
a below-threshold chunk is rejected. This uses the simplest possible
parser -- reading a score straight out of a dict -- so the guardrail itself
is shown working before any of the harder JSON-repair machinery exists.

In [ ]:
SCORE_THRESHOLD = 5

_off_topic_chunk = {"chunk_id": "guardrail-demo", "title": "Off-topic excerpt", "rcs_score": 2, "summary": "Not relevant to the query."}

_kept = _off_topic_chunk["summary"] and _off_topic_chunk["rcs_score"] >= SCORE_THRESHOLD
print(f"chunk {_off_topic_chunk['chunk_id']!r} scored {_off_topic_chunk['rcs_score']}/10 "
      f"(threshold={SCORE_THRESHOLD}) -- kept: {_kept}")
assert not _kept, "the guardrail must reject a chunk scoring below threshold"
print("guardrail confirmed: a below-threshold chunk is rejected, before any parsing ladder is built")

## Step 2 — the parsing/repair ladder

An LLM asked to return JSON does not always return valid JSON. RCS handles
this with a ladder, tried in order:

1. Strip a ` ```json ... ``` ` fence if the model wrapped its answer in one.
2. Parse as JSON directly.
3. If that fails, repair a common defect (a trailing comma before `}`/`]`)
   and parse again.
4. If it's still not valid JSON, fall back to a **legacy plain-text format**:
   scan lines in reverse for a bare `0-10` integer or an `N/10` fraction, and
   treat everything else as the summary.

Every one of these steps is real, load-bearing logic -- not a simplified
stand-in -- because the parsing ladder itself, not just the LLM call, is
what makes RCS resilient to a model that doesn't follow instructions.

In [ ]:
import json
import re

_JSON_FENCE_RE = re.compile(r"```(?:json)?\s*([\s\S]*?)\s*```", re.IGNORECASE)
_SCORE_LINE_RE = re.compile(r"^\s*(\d{1,2})\s*$")
_FRACTION_RE = re.compile(r"(\d{1,2})\s*/\s*10")
_TRAILING_COMMA_RE = re.compile(r",\s*([}\]])")


def _strip_json_fences(text: str) -> str:
    t = (text or "").strip()
    m = _JSON_FENCE_RE.search(t)
    return m.group(1).strip() if m else t


def _repair_json(text: str) -> str:
    return _TRAILING_COMMA_RE.sub(r"\1", text)


def _parse_json_rcs_output(text_out: str):
    raw = _strip_json_fences(text_out)
    start = raw.find("{")
    end = raw.rfind("}")
    if start >= 0 and end > start:
        raw = raw[start : end + 1]
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        try:
            obj = json.loads(_repair_json(raw))
        except json.JSONDecodeError:
            return None
    if not isinstance(obj, dict):
        return None
    summary = str(obj.get("summary") or obj.get("Summary") or "").strip()
    score_raw = obj.get("relevance_score", obj.get("relevanceScore", obj.get("score")))
    try:
        score = int(float(score_raw))
    except (TypeError, ValueError):
        score = 0
    score = max(0, min(10, score))
    evidence_level = str(obj.get("evidence_level") or obj.get("evidenceLevel") or "").strip() or "Unknown"
    return summary, score, evidence_level


def _extract_summary_and_score_legacy(text_out: str):
    lines = (text_out or "").strip().splitlines()
    score = 0
    for line in reversed(lines):
        m = _SCORE_LINE_RE.match(line)
        if m:
            score = max(0, min(10, int(m.group(1))))
            break
        frac = _FRACTION_RE.search(line)
        if frac:
            score = max(0, min(10, int(frac.group(1))))
            break
    filtered = [
        line for line in lines if not _SCORE_LINE_RE.match(line) and line.strip().lower() != "not applicable"
    ]
    return "\n".join(filtered).strip(), score, "Unknown"


def _parse_rcs_response(text_out: str):
    parsed = _parse_json_rcs_output(text_out)
    if parsed is not None:
        return parsed
    return _extract_summary_and_score_legacy(text_out)

## Step 3 — exercise the ladder on both well-formed and malformed output

Five canned strings, standing in for five different ways a real LLM response
comes back: clean JSON, fenced JSON, JSON with a trailing comma, the legacy
plain-text format, and something a model returns when it just refuses to
follow the schema.

In [ ]:
sample_responses = {
    "clean_json": (
        '{"summary": "Directly reports NPWT healing-rate outcomes.", '
        '"relevance_score": 8, "evidence_level": "Level II"}'
    ),
    "fenced_json": (
        "```json\n"
        '{"summary": "Background context on wound staging, not directly on point.", '
        '"relevance_score": 4, "evidence_level": "Level V"}\n'
        "```"
    ),
    "trailing_comma_json": (
        '{"summary": "RCT comparing NPWT to standard dressings.", '
        '"relevance_score": 7, "evidence_level": "Level II",}'
    ),
    "legacy_plain_text": (
        "This excerpt discusses amputation risk reduction strategies, tangential to "
        "the question asked.\n6"
    ),
    "unparseable_garbage": "I cannot assess this excerpt without more context.",
}

rows = []
for label, text_out in sample_responses.items():
    summary, score, evidence_level = _parse_rcs_response(text_out)
    rows.append((label, score, evidence_level, summary[:45]))

nbio.table(rows, headers=("sample", "score", "evidence_level", "summary"))

## Step 4 — the real filter, over a full candidate set

`rcs_rerank`'s actual job: score every candidate with the real parsing
ladder from Step 2, keep the ones at or above `score_threshold`, and
silently -- structurally -- drop the rest. "Silently" is the point being
made here, not a criticism in passing: nothing downstream of this filter
ever sees a below-threshold chunk again, so if that chunk was the only
correct source for a narrow question, the retrieval pipeline now looks
unanswerable rather than merely under-scored. This is the same guardrail
proven in Step 1, now applied through the full ladder to five realistic
candidates.

In [ ]:
SCORE_THRESHOLD = 5

candidate_chunks = [
    {"chunk_id": "c1", "title": "NPWT RCT in diabetic foot ulcers", "llm_response": sample_responses["clean_json"]},
    {"chunk_id": "c2", "title": "Pressure injury staging background", "llm_response": sample_responses["fenced_json"]},
    {
        "chunk_id": "c3",
        "title": "NPWT vs standard dressing RCT (second source)",
        "llm_response": sample_responses["trailing_comma_json"],
    },
    {
        "chunk_id": "c4",
        "title": "Amputation risk reduction strategies",
        "llm_response": sample_responses["legacy_plain_text"],
    },
    {"chunk_id": "c5", "title": "Off-topic excerpt", "llm_response": sample_responses["unparseable_garbage"]},
]

scored = []
for c in candidate_chunks:
    summary, score, evidence_level = _parse_rcs_response(c["llm_response"])
    scored.append({**c, "summary": summary, "rcs_score": score, "evidence_level": evidence_level})

kept = [c for c in scored if c["summary"] and c["rcs_score"] >= SCORE_THRESHOLD]
dropped = [c for c in scored if c not in kept]

nbio.delta(candidate_chunks, kept, label="RCS filter (threshold=5)")
print()
nbio.table(
    [(c["chunk_id"], c["rcs_score"], c["title"][:45]) for c in kept], headers=("chunk_id", "rcs_score", "title")
)
print("\ndropped -- never reach the ranked list:")
nbio.table(
    [(c["chunk_id"], c["rcs_score"], c["title"][:45]) for c in dropped], headers=("chunk_id", "rcs_score", "title")
)

## Step 5 — batch mode — `_parse_json_rcs_batch` / `_chunk_from_batch_entry`

RCS also supports a batch mode (`RCS_MODE=batch`): score several chunks in
one LLM call (a JSON array keyed by `id`) instead of one call per chunk, to
cut latency and cost. The parsing side of that is implemented here; the
batching and concurrency-limiting machinery around it
(`_rcs_rerank_batched`, telemetry hooks) is orchestration plumbing that
doesn't need to be included for the logic to be clear.

In [ ]:
def _parse_json_rcs_batch(text_out: str):
    raw = _strip_json_fences(text_out)
    start = raw.find("[")
    end = raw.rfind("]")
    if start < 0 or end <= start:
        return None
    raw = raw[start : end + 1]
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        try:
            obj = json.loads(_repair_json(raw))
        except json.JSONDecodeError:
            return None
    return [item for item in obj if isinstance(item, dict)] if isinstance(obj, list) else None


def _chunk_from_batch_entry(chunk: dict, entry: dict, score_threshold: int):
    summary = str(entry.get("summary") or "").strip()
    try:
        score = int(float(entry.get("relevance_score", entry.get("score"))))
    except (TypeError, ValueError):
        score = 0
    score = max(0, min(10, score))
    evidence_level = str(entry.get("evidence_level") or entry.get("evidenceLevel") or "Unknown").strip()
    if summary and score >= int(score_threshold):
        return {**chunk, "summary": summary, "rcs_score": score, "evidence_level": evidence_level or "Unknown"}
    return None


batch_response = "\n".join(
    [
        "[",
        '  {"id": 0, "summary": "Directly on point.", "relevance_score": 9, "evidence_level": "Level I"},',
        '  {"id": 1, "summary": "Tangential background.", "relevance_score": 3, "evidence_level": "Level IV"}',
        "]",
    ]
)

batch_chunks = {0: {"chunk_id": "b0", "title": "Meta-analysis of NPWT trials"}, 1: {"chunk_id": "b1", "title": "General wound-care overview"}}

parsed_batch = _parse_json_rcs_batch(batch_response)
batch_kept = [
    _chunk_from_batch_entry(batch_chunks[int(e["id"])], e, SCORE_THRESHOLD) for e in parsed_batch
]
batch_kept = [c for c in batch_kept if c is not None]
nbio.table(
    [(c["chunk_id"], c["rcs_score"], c["title"][:45]) for c in batch_kept], headers=("chunk_id", "rcs_score", "title")
)

## Step 6 — live scoring under a spend ceiling — only runs if a key is loaded

The product calls a configured LLM client (`get_rcs_client()` in `rcs.py`, wired through `llm_client.py`/`prompt_manager.py`). Those two files are provider/prompt-template plumbing specific to the product; the generic version below picks whichever of Groq or OpenAI has a key loaded and asks it the same question RCS asks: score this excerpt's relevance to this query, 0-10, plus a one-line summary and an evidence level, as JSON. This is the one real paid call in this notebook, so it runs inside `nbio.cost_meter` — RCS scores many chunks per query in the real product, which is exactly the shape a ceiling protects against a runaway loop.

In [ ]:
import os

RCS_PROMPT = (
    "Score how relevant the following excerpt is to the question, from 0 (irrelevant) "
    "to 10 (directly answers it). Also assign an Oxford CEBM evidence level "
    "(Level I-V, or Unknown). Respond with JSON only: "
    '{{"summary": "<one line>", "relevance_score": <0-10>, "evidence_level": "<Level I-V|Unknown>"}}\n\n'
    "Question: {question}\n\nExcerpt ({title}):\n{text}"
)

_LAST_USAGE: dict = {}


def _get_llm_client():
    if os.environ.get("GROQ_API_KEY"):
        from groq import Groq

        return "groq", Groq(api_key=os.environ["GROQ_API_KEY"])
    if os.environ.get("OPENAI_API_KEY"):
        from openai import OpenAI

        return "openai", OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    return None, None


def score_chunk_live(question: str, title: str, text: str, provider: str, client) -> dict:
    prompt = RCS_PROMPT.format(question=question, title=title, text=text[:2000])
    model_id = "llama-3.1-8b-instant" if provider == "groq" else "gpt-4o-mini"
    resp = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=200,
    )
    text_out = resp.choices[0].message.content
    usage = getattr(resp, "usage", None)
    _LAST_USAGE["model"] = model_id
    _LAST_USAGE["prompt_tokens"] = getattr(usage, "prompt_tokens", 0) if usage else 0
    _LAST_USAGE["completion_tokens"] = getattr(usage, "completion_tokens", 0) if usage else 0
    summary, score, evidence_level = _parse_rcs_response(text_out)
    return {"summary": summary, "rcs_score": score, "evidence_level": evidence_level}


provider, client = _get_llm_client()
with nbio.cost_meter(budget_usd=0.50) as meter:
    if provider is None:
        print(
            "No GROQ_API_KEY or OPENAI_API_KEY loaded (see nbio.show_environment() above) -- "
            "skipping live scoring. The offline parser/threshold demo above already exercises "
            "the full RCS parsing, repair, and filtering logic without a key."
        )
    else:
        print(f"scoring live with provider={provider!r}")
        live_result = score_chunk_live(
            question="What is the evidence for NPWT in diabetic foot ulcers?",
            title="NPWT RCT in diabetic foot ulcers",
            text="A randomized controlled trial found NPWT accelerated wound closure "
            "compared to standard dressings in diabetic foot ulcers.",
            provider=provider,
            client=client,
        )
        meter.record(_LAST_USAGE["model"], _LAST_USAGE["prompt_tokens"], _LAST_USAGE["completion_tokens"])
        nbio.show_json(live_result)

print()
print(meter.report())


## Wrap-up

RCS's real cost is not the LLM call latency -- it's the chunks that never
make it past `score_threshold`. In the offline demo above, 2 of 5 candidates
were dropped (one deliberately off-topic, one that failed to parse into a
usable summary at all); in the real pipeline, a below-threshold chunk is
gone before ranking ever sees it, and nothing downstream can tell the
difference between "no evidence exists" and "evidence existed and scored 4".

Next: `05-ranking-and-final-score.ipynb` -- how the survivors of this filter
get ranked, and why the five score components attached at ranking time are
worth keeping around afterward.